# 📦 Azure Subscription Resource Inventory

This notebook **auto-discovers** every resource across one or more Azure
subscriptions and produces a **comprehensive Excel workbook** with:

- **One tab per subscription** listing every resource (name, type, location, resource group, tags)
- **A summary tab** with counts of each resource type aggregated across all subscriptions

| Feature | Details |
|---------|--------|
| Resource Discovery | Azure Resource Graph (`resources` table) – returns all ARM resources |
| Multi-Subscription | Scans a user-supplied list of subscription IDs (or auto-discovers all accessible) |
| Excel Export | One sheet per subscription + a cross-subscription summary sheet |
| Rate-Limit Handling | Exponential backoff with jitter, Retry-After header support |
| Pagination | Handles Resource Graph `$skipToken` for subscriptions with thousands of resources |

```mermaid
flowchart LR
    A["\ud83d\udd10 Azure Auth (DefaultAzureCredential)"] --> B["\ud83d\udce1 Resource Graph Client"]
    B --> C["\ud83c\udfe2 List Subscriptions"]
    C --> D["\ud83d\udce6 Query Resources per Sub"]
    D --> E["\ud83d\udcca Build DataFrames"]
    E --> F["\ud83d\udcbe Export Excel (tab per sub + summary)"]
```

## Prerequisites

### \ud83d\udd10 Required Permissions

| Scope | Required Role | Notes |
|-------|--------------|-------|
| Each subscription to scan | **Reader** (minimum) | Any role with `Microsoft.Resources/subscriptions/resources/read` |
| Azure Resource Graph | No additional role | Available to any user with resource read access |

### \ud83d\udccb Python Packages

- `azure-identity` – authentication via `DefaultAzureCredential`
- `azure-mgmt-resourcegraph` – Azure Resource Graph queries
- `azure-mgmt-resource` – subscription discovery
- `pandas` – data manipulation
- `openpyxl` – Excel export

### \u23f1\ufe0f API Limits & Throttling

| API | Limit | Handling |
|-----|-------|----------|
| Resource Graph (per tenant) | 15 requests / 5 seconds | Auto-throttle with sliding window |
| Resource Graph (per query) | 1000 rows max per page | Automatic pagination via `$skipToken` |
| ARM Management | 12000 reads / hour | Tracked; unlikely to hit for listing |
| HTTP 429 / 5xx | Retryable | Exponential backoff up to 5 min, honours `Retry-After` |

## Key References

| Topic | Link |
|-------|------|
| Azure Resource Graph overview | https://learn.microsoft.com/en-us/azure/governance/resource-graph/overview |
| Resource Graph query language | https://learn.microsoft.com/en-us/azure/governance/resource-graph/concepts/query-language |
| Throttling guidance | https://learn.microsoft.com/en-us/azure/governance/resource-graph/concepts/guidance-for-throttled-requests |
| ARM rate limits | https://learn.microsoft.com/en-us/azure/azure-resource-manager/management/request-limits-and-throttling |

## 0️⃣ Install Dependencies & Setup

In [ ]:
# ── Install dependencies (safe to re-run) ────────────────────────────
import subprocess
import sys

packages = [
    "azure-identity",
    "azure-mgmt-resourcegraph",
    "azure-mgmt-resource",
    "azure-mgmt-subscription",
    "pandas",
    "openpyxl",
]

for pkg in packages:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", pkg],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

print("\u2705 All dependencies installed")

✅ All dependencies installed


In [6]:
# ── Imports & Core Setup ──────────────────────────────────────────────
import time
import random
import os
from collections import deque

import pandas as pd
from azure.identity import DefaultAzureCredential
from azure.mgmt.resourcegraph import ResourceGraphClient
from azure.mgmt.resourcegraph.models import (
    QueryRequest,
    QueryRequestOptions,
    ResultFormat,
)
from azure.mgmt.subscription import SubscriptionClient

# ── Authenticate ──────────────────────────────────────────────────────
# DefaultAzureCredential tries (in order): env vars, managed identity,
# Azure CLI, VS Code, Azure PowerShell, interactive browser.
credential = DefaultAzureCredential()
graph_client = ResourceGraphClient(credential)
sub_client = SubscriptionClient(credential)

print("\u2705 Azure authentication successful")
print("   Strategy: DefaultAzureCredential (CLI / Managed Identity / VS Code / Browser)")

✅ Azure authentication successful
   Strategy: DefaultAzureCredential (CLI / Managed Identity / VS Code / Browser)


## 1️⃣ Rate-Limit Tracker & Resilient Query Helper

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Rate-Limit Tracker – prevents hitting Resource Graph throttle limits
# Resource Graph: 15 requests per 5-second window (per tenant)
# ═══════════════════════════════════════════════════════════════════════


class RateLimitTracker:
    """
    Sliding-window rate tracker for Azure Resource Graph.
    - 15 requests per 5s window (per-tenant limit).
    - Auto-waits when nearing the cap.
    """

    def __init__(self, max_calls=15, window_secs=5, warn_pct=0.80):
        self.max_calls = max_calls
        self.window = window_secs
        self.warn_pct = warn_pct
        self.timestamps: deque = deque()
        self.total_calls = 0
        self.total_retries = 0
        self.total_wait_secs = 0.0
        self.start_time = time.time()

    def _prune(self):
        cutoff = time.time() - self.window
        while self.timestamps and self.timestamps[0] < cutoff:
            self.timestamps.popleft()

    def wait_if_needed(self):
        """Block until a request slot is available."""
        self._prune()
        if len(self.timestamps) >= self.max_calls:
            wait = self.timestamps[0] + self.window - time.time() + 0.1
            if wait > 0:
                print(f"   \u23f3 Rate limit: {len(self.timestamps)}/{self.max_calls} "
                      f"in {self.window}s window. Pausing {wait:.1f}s ...")
                self.total_wait_secs += wait
                time.sleep(wait)
                self._prune()
        elif len(self.timestamps) >= int(self.max_calls * self.warn_pct):
            remaining = self.max_calls - len(self.timestamps)
            print(f"   \u26a0\ufe0f  Rate limit: {len(self.timestamps)}/{self.max_calls} "
                  f"used ({remaining} remaining in window)")

    def record_call(self):
        self.wait_if_needed()
        self.timestamps.append(time.time())
        self.total_calls += 1

    def record_retry(self, wait_secs):
        self.total_retries += 1
        self.total_wait_secs += wait_secs

    def summary(self):
        elapsed = (time.time() - self.start_time) / 60
        return (
            f"   API calls: {self.total_calls} total | "
            f"Retries: {self.total_retries} | "
            f"Wait time: {self.total_wait_secs:.0f}s | "
            f"Elapsed: {elapsed:.1f} min"
        )


rate_tracker = RateLimitTracker(max_calls=15, window_secs=5)


# ═══════════════════════════════════════════════════════════════════════
# Resilient Resource Graph Query Helper
# ═══════════════════════════════════════════════════════════════════════


def query_resource_graph(
    client,
    query: str,
    subscription_ids: list,
    max_retries: int = 5,
    base_delay: float = 2.0,
    max_delay: float = 300.0,
) -> list:
    """
    Execute a Resource Graph query with:
    - Automatic pagination ($skipToken)
    - Sliding-window rate-limit tracking
    - Exponential backoff with jitter for 429 / 5xx
    - Retry-After header support

    Returns a flat list of row dicts.
    """
    all_rows = []
    skip_token = None
    page = 0

    while True:
        options = QueryRequestOptions(
            result_format=ResultFormat.OBJECT_ARRAY,
            top=1000,
        )
        if skip_token:
            options.skip_token = skip_token

        request = QueryRequest(
            subscriptions=subscription_ids,
            query=query,
            options=options,
        )

        for attempt in range(max_retries + 1):
            try:
                rate_tracker.record_call()
                response = client.resources(request)
                break  # success
            except Exception as e:
                error_msg = str(e)
                is_throttle = "429" in error_msg or "ThrottledRequest" in error_msg
                is_transient = any(
                    code in error_msg for code in ["500", "502", "503", "504"]
                )

                if (is_throttle or is_transient) and attempt < max_retries:
                    # Try to parse Retry-After
                    wait = base_delay * (2 ** attempt)
                    if is_throttle and hasattr(e, "response"):
                        resp = getattr(e, "response")
                        if resp is not None:
                            retry_after = resp.headers.get("Retry-After")
                            if retry_after:
                                wait = float(retry_after)
                    # Add jitter
                    wait = min(wait * (0.75 + random.random() * 0.5), max_delay)
                    print(
                        f"   \u23f3 {'Throttled (429)' if is_throttle else 'Transient error'}"
                        f" \u2013 retrying in {wait:.1f}s "
                        f"(attempt {attempt + 1}/{max_retries})"
                    )
                    rate_tracker.record_retry(wait)
                    time.sleep(wait)
                else:
                    raise

        rows = response.data
        all_rows.extend(rows)
        page += 1

        skip_token = response.skip_token
        if not skip_token:
            break

        if page % 5 == 0:
            print(f"      ... fetched {len(all_rows)} resources so far (page {page}) ...")

    return all_rows


def progress_bar(current, total, label="", width=30):
    """Simple text progress bar."""
    pct = current / total if total > 0 else 0
    filled = int(width * pct)
    bar = "\u2588" * filled + "\u2591" * (width - filled)
    print(f"\r   [{bar}] {current}/{total} ({pct:.0%}) {label}", end="", flush=True)


print("\u2705 Rate-limit tracker initialized (15 req / 5s window)")
print("\u2705 Resource Graph query helper loaded (pagination + backoff)")

✅ Rate-limit tracker initialized (15 req / 5s window)
✅ Resource Graph query helper loaded (pagination + backoff)


## 2️⃣ Configuration

Provide a list of subscription IDs to scan, or leave empty to **auto-discover**
all subscriptions accessible by the current credential.

In [ ]:
# ── Configuration ────────────────────────────────────────────────────

# Option 1: Explicitly list subscription IDs to scan
# SUBSCRIPTION_IDS = [
#     "aaaaaaaa-bbbb-cccc-dddd-eeeeeeeeeeee",
#     "ffffffff-1111-2222-3333-444444444444",
# ]

# Option 2: Leave empty to auto-discover all accessible subscriptions
SUBSCRIPTION_IDS = []

# Output filename for the Excel workbook
OUTPUT_FILE = "azure-resource-inventory.xlsx"

# Maximum subscriptions to process (safety limit)
MAX_SUBSCRIPTIONS = 100

# ── Display config ────────────────────────────────────────────────────
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_colwidth", 60)

print("\u2705 Azure Subscription Resource Inventory")
print(f"   Subscriptions:    {'User-provided (' + str(len(SUBSCRIPTION_IDS)) + ')' if SUBSCRIPTION_IDS else 'Auto-discover all'}")
print(f"   Max Subscriptions: {MAX_SUBSCRIPTIONS}")
print(f"   Output File:      {OUTPUT_FILE}")

✅ Azure Subscription Resource Inventory
   Subscriptions:    Auto-discover all
   Max Subscriptions: 100
   Output File:      azure-resource-inventory.xlsx


## 3️⃣ Discover Subscriptions

If no subscription IDs are provided, the notebook auto-discovers all
subscriptions the current credential has access to.

In [9]:
# ── Discover subscriptions ────────────────────────────────────────────

if SUBSCRIPTION_IDS:
    # Validate user-provided subscription IDs by fetching details
    subscriptions = []
    for sid in SUBSCRIPTION_IDS:
        try:
            sub = sub_client.subscriptions.get(sid)
            subscriptions.append({
                "subscription_id": sub.subscription_id,
                "display_name": sub.display_name,
                "state": str(sub.state) if sub.state else "Unknown",
            })
        except Exception as e:
            print(f"   \u26a0\ufe0f  Could not access subscription {sid}: {e}")
else:
    # Auto-discover all accessible subscriptions
    print("\u27a4 Auto-discovering subscriptions ...")
    subscriptions = []
    for sub in sub_client.subscriptions.list():
        if sub.state and str(sub.state) == "Enabled":
            subscriptions.append({
                "subscription_id": sub.subscription_id,
                "display_name": sub.display_name,
                "state": str(sub.state),
            })

# Safety cap
if len(subscriptions) > MAX_SUBSCRIPTIONS:
    print(f"\u26a0\ufe0f  Found {len(subscriptions)} subscriptions, capping at {MAX_SUBSCRIPTIONS}.")
    print("   Increase MAX_SUBSCRIPTIONS in the Configuration cell if needed.")
    subscriptions = subscriptions[:MAX_SUBSCRIPTIONS]

df_subscriptions = pd.DataFrame(subscriptions)

print(f"\n\u25b6 Found {len(subscriptions)} subscription(s)")
print()
if not df_subscriptions.empty:
    print(df_subscriptions[["display_name", "subscription_id", "state"]].to_string(index=False))
else:
    print("\u274c No accessible subscriptions found. Check your credentials and permissions.")

➤ Auto-discovering subscriptions ...

▶ Found 2 subscription(s)

                         display_name                      subscription_id   state
MCAPS-Hybrid-REQ-137206-2025-ncheruvu 64e1939f-6460-4656-ad75-dcc277b155f1 Enabled
MCAPS-Hybrid-REQ-115969-2025-huntleyh 3ce66af2-55dc-48a9-8498-d973bda26aa5 Enabled


## 4️⃣ Query Resources Across All Subscriptions

Uses Azure Resource Graph to query all resources per subscription.
Results are paginated (1000 rows per page) and rate-limited automatically.

In [10]:
# ── Query resources per subscription ──────────────────────────────────

RESOURCE_QUERY = """
resources
| project
    name,
    type,
    resourceGroup,
    location,
    subscriptionId,
    sku = tostring(sku.name),
    kind,
    tags,
    id
| order by type asc, name asc
"""

all_resources = {}  # subscription_id -> list of resource dicts
errors = {}         # subscription_id -> error message
total_subs = len(subscriptions)

print(f"\u27a4 Querying resources across {total_subs} subscription(s) ...\n")
scan_start = time.time()

for idx, sub in enumerate(subscriptions):
    sub_id = sub["subscription_id"]
    sub_name = sub["display_name"]
    progress_bar(idx, total_subs, label=sub_name)

    try:
        rows = query_resource_graph(
            client=graph_client,
            query=RESOURCE_QUERY,
            subscription_ids=[sub_id],
        )
        all_resources[sub_id] = rows
        print(f"\n   \u2705 {sub_name}: {len(rows)} resources")
    except Exception as e:
        errors[sub_id] = str(e)
        all_resources[sub_id] = []
        print(f"\n   \u274c {sub_name}: Error - {e}")

progress_bar(total_subs, total_subs, label="Done!")

elapsed = time.time() - scan_start
total_resources = sum(len(v) for v in all_resources.values())

print(f"\n\n\u25b6 Scan complete in {elapsed:.1f}s")
print(f"   Total resources found: {total_resources:,}")
print(f"   Subscriptions scanned: {total_subs - len(errors)}/{total_subs} successful")
if errors:
    print(f"   \u26a0\ufe0f  {len(errors)} subscription(s) had errors")
print()
print(rate_tracker.summary())

➤ Querying resources across 2 subscription(s) ...

   [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░] 0/2 (0%) MCAPS-Hybrid-REQ-137206-2025-ncheruvu
   ✅ MCAPS-Hybrid-REQ-137206-2025-ncheruvu: 144 resources
   [███████████████░░░░░░░░░░░░░░░] 1/2 (50%) MCAPS-Hybrid-REQ-115969-2025-huntleyh
   ✅ MCAPS-Hybrid-REQ-115969-2025-huntleyh: 1 resources
   [██████████████████████████████] 2/2 (100%) Done!

▶ Scan complete in 5.3s
   Total resources found: 145
   Subscriptions scanned: 2/2 successful

   API calls: 2 total | Retries: 0 | Wait time: 0s | Elapsed: 0.6 min


## 5️⃣ Build DataFrames & Preview

Convert raw results into clean DataFrames – one per subscription – and
build the cross-subscription summary.

In [11]:
# ── Build per-subscription DataFrames ─────────────────────────────────

sub_name_map = {s["subscription_id"]: s["display_name"] for s in subscriptions}
sub_dataframes = {}  # display_name -> DataFrame

for sub_id, resources in all_resources.items():
    sub_name = sub_name_map.get(sub_id, sub_id)
    if not resources:
        sub_dataframes[sub_name] = pd.DataFrame(
            columns=["name", "type", "resourceGroup", "location", "sku", "kind", "tags"]
        )
        continue

    df = pd.DataFrame(resources)

    # Flatten tags dict to a readable string
    if "tags" in df.columns:
        df["tags"] = df["tags"].apply(
            lambda t: "; ".join(f"{k}={v}" for k, v in t.items())
            if isinstance(t, dict) and t
            else ""
        )

    # Clean up type names for readability (e.g. microsoft.compute/virtualmachines -> Virtual Machines)
    if "type" in df.columns:
        df["service"] = df["type"].apply(lambda t: t.split("/")[0] if "/" in str(t) else str(t))
        df["resource_type"] = df["type"].apply(lambda t: t.split("/", 1)[1] if "/" in str(t) else str(t))

    # Select and order columns
    cols = ["name", "type", "service", "resource_type", "resourceGroup",
            "location", "sku", "kind", "tags"]
    cols = [c for c in cols if c in df.columns]
    df = df[cols].sort_values(["type", "name"]).reset_index(drop=True)

    sub_dataframes[sub_name] = df
    print(f"\u25b8 {sub_name}: {len(df)} resources, {df['type'].nunique()} resource types")

print(f"\n\u2705 Built {len(sub_dataframes)} subscription DataFrames")

▸ MCAPS-Hybrid-REQ-137206-2025-ncheruvu: 144 resources, 35 resource types
▸ MCAPS-Hybrid-REQ-115969-2025-huntleyh: 1 resources, 1 resource types

✅ Built 2 subscription DataFrames


In [12]:
# ── Build cross-subscription summary ──────────────────────────────────

summary_rows = []

for sub_name, df in sub_dataframes.items():
    if df.empty:
        continue
    counts = df.groupby(["service", "resource_type"]).size().reset_index(name="count")
    counts["subscription"] = sub_name
    summary_rows.append(counts)

if summary_rows:
    df_detail = pd.concat(summary_rows, ignore_index=True)

    # Pivot: rows = (service, resource_type), columns = subscription, values = count
    df_summary_pivot = df_detail.pivot_table(
        index=["service", "resource_type"],
        columns="subscription",
        values="count",
        aggfunc="sum",
        fill_value=0,
    )
    df_summary_pivot["TOTAL"] = df_summary_pivot.sum(axis=1)
    df_summary_pivot = df_summary_pivot.sort_values("TOTAL", ascending=False)

    # Flat summary: total counts per resource type across all subs
    df_summary_flat = (
        df_detail.groupby(["service", "resource_type"])["count"]
        .sum()
        .reset_index()
        .sort_values("count", ascending=False)
        .reset_index(drop=True)
    )
    df_summary_flat.columns = ["Service (Provider)", "Resource Type", "Total Count"]

    print("\u25b6 Cross-Subscription Resource Summary (Top 20)")
    print("=" * 80)
    print(df_summary_flat.head(20).to_string(index=False))
    print(f"\n   Total unique resource types: {len(df_summary_flat)}")
    print(f"   Grand total resources:       {df_summary_flat['Total Count'].sum():,}")
else:
    df_summary_pivot = pd.DataFrame()
    df_summary_flat = pd.DataFrame()
    print("\u26a0\ufe0f  No resources found across any subscription.")

▶ Cross-Subscription Resource Summary (Top 20)
               Service (Provider)                       Resource Type  Total Count
                    microsoft.app                       containerapps           15
    microsoft.operationalinsights                          workspaces           12
              microsoft.eventgrid                        systemtopics           11
                microsoft.storage                     storageaccounts           11
      microsoft.cognitiveservices                            accounts            8
        microsoft.managedidentity              userassignedidentities            6
       microsoft.alertsmanagement             smartdetectoralertrules            6
microsoft.machinelearningservices                          workspaces            6
               microsoft.insights                          components            6
                microsoft.network                     networkwatchers            6
                    microsoft.web       

## 6️⃣ Export to Excel

Creates an Excel workbook with:
- **Summary** tab – resource counts per type across all subscriptions
- **Summary (Pivot)** tab – pivot table with subscriptions as columns
- **One tab per subscription** – full resource listing

In [13]:
# ── Export Excel workbook ─────────────────────────────────────────────
from openpyxl.utils import get_column_letter


def sanitize_sheet_name(name: str) -> str:
    """Excel sheet names: max 31 chars, no [ ] : * ? / \\ characters."""
    for ch in ["[", "]", ":", "*", "?", "/", "\\"]:
        name = name.replace(ch, "_")
    return name[:31]


def auto_fit_columns(writer, sheet_name, df):
    """Auto-fit column widths based on content (capped at 60)."""
    worksheet = writer.sheets[sheet_name]
    for i, col in enumerate(df.columns, 1):
        max_len = max(
            df[col].astype(str).map(len).max() if not df.empty else 0,
            len(str(col)),
        )
        worksheet.column_dimensions[get_column_letter(i)].width = min(max_len + 2, 60)


# Create workbook
with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    # 1. Summary tab (flat)
    if not df_summary_flat.empty:
        df_summary_flat.to_excel(writer, sheet_name="Summary", index=False)
        auto_fit_columns(writer, "Summary", df_summary_flat)
        print("\u25b8 Summary tab written")

    # 2. Summary pivot tab
    if not df_summary_pivot.empty:
        df_pivot_reset = df_summary_pivot.reset_index()
        df_pivot_reset.to_excel(writer, sheet_name="Summary (Pivot)", index=False)
        auto_fit_columns(writer, "Summary (Pivot)", df_pivot_reset)
        print("\u25b8 Summary (Pivot) tab written")

    # 3. One tab per subscription
    used_names = set()
    for sub_name, df in sub_dataframes.items():
        sheet = sanitize_sheet_name(sub_name)
        # Handle duplicate sheet names
        base_sheet = sheet
        suffix = 1
        while sheet in used_names:
            sheet = f"{base_sheet[:28]}_{suffix}"
            suffix += 1
        used_names.add(sheet)

        df.to_excel(writer, sheet_name=sheet, index=False)
        auto_fit_columns(writer, sheet, df)
        print(f"   \u25b8 {sheet}: {len(df)} rows")

print(f"\n\u2705 Workbook saved: {os.path.abspath(OUTPUT_FILE)}")
print(f"   Sheets: {2 + len(sub_dataframes)} (Summary + Pivot + {len(sub_dataframes)} subscriptions)")

▸ Summary tab written
▸ Summary (Pivot) tab written
   ▸ MCAPS-Hybrid-REQ-137206-2025-nc: 144 rows
   ▸ MCAPS-Hybrid-REQ-115969-2025-hu: 1 rows

✅ Workbook saved: c:\Git\AZ\msft-fabric-utils\azure-resource-inventory\azure-resource-inventory.xlsx
   Sheets: 4 (Summary + Pivot + 2 subscriptions)


## 7️⃣ Final Summary & Statistics

In [14]:
# ── Final Summary ─────────────────────────────────────────────────────

print("\u25b6 AZURE RESOURCE INVENTORY \u2013 FINAL SUMMARY")
print("=" * 80)
print(f"   Subscriptions scanned:     {len(subscriptions)}")
print(f"   Subscriptions with errors: {len(errors)}")
print(f"   Total resources found:     {total_resources:,}")
print()

# Per-subscription breakdown
print("   PER-SUBSCRIPTION BREAKDOWN")
print("   " + "-" * 76)
print(f"   {'Subscription':<40} {'Resources':>10} {'Types':>8} {'Status':>10}")
print("   " + "-" * 76)
for sub in subscriptions:
    sid = sub["subscription_id"]
    sname = sub["display_name"][:38]
    rcount = len(all_resources.get(sid, []))
    sdf = sub_dataframes.get(sub["display_name"], pd.DataFrame())
    tcount = sdf["type"].nunique() if not sdf.empty and "type" in sdf.columns else 0
    status = "\u274c Error" if sid in errors else "\u2705 OK"
    print(f"   {sname:<40} {rcount:>10,} {tcount:>8} {status:>10}")

print("   " + "-" * 76)
print()

# Top services
if not df_summary_flat.empty:
    print("   TOP 10 SERVICES BY RESOURCE COUNT")
    print("   " + "-" * 60)
    svc_summary = (
        df_summary_flat.groupby("Service (Provider)")["Total Count"]
        .sum()
        .sort_values(ascending=False)
        .head(10)
    )
    for svc, count in svc_summary.items():
        print(f"   {svc:<45} {count:>8,}")
    print()

print(f"\u25b8 Output file: {os.path.abspath(OUTPUT_FILE)}")
print()
print(rate_tracker.summary())
print("\n\u2705 Inventory complete!")

▶ AZURE RESOURCE INVENTORY – FINAL SUMMARY
   Subscriptions scanned:     2
   Subscriptions with errors: 0
   Total resources found:     145

   PER-SUBSCRIPTION BREAKDOWN
   ----------------------------------------------------------------------------
   Subscription                              Resources    Types     Status
   ----------------------------------------------------------------------------
   MCAPS-Hybrid-REQ-137206-2025-ncheruvu           144       35       ✅ OK
   MCAPS-Hybrid-REQ-115969-2025-huntleyh             1        1       ✅ OK
   ----------------------------------------------------------------------------

   TOP 10 SERVICES BY RESOURCE COUNT
   ------------------------------------------------------------
   microsoft.network                                   21
   microsoft.app                                       19
   microsoft.eventgrid                                 13
   microsoft.web                                       13
   microsoft.operationalinsig